In [ ]:
# Importing libraries

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Reading in dataset

df = pd.read_csv('final_cohort.csv')

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
# Dropping isoforms- not directly applicable to MR

proteins_to_drop = ['Paired immunoglobulin-like type 2 receptor alpha isoform FDF03-M14 (PILRA)' ,
                    'Apolipoprotein E (isoform E3) (APOE)', 'Apolipoprotein E (isoform E4) (APOE)',
                    'Apolipoprotein E (isoform E2) (APOE)', 
                    'Paired immunoglobulin-like type 2 receptor alpha isoform FDF03-deltaTM (PILRA)']

In [ ]:
df_no_isoforms = df.drop(columns= proteins_to_drop)

In [ ]:
df_no_isoforms.head()

In [ ]:
df_no_isoforms.columns

In [ ]:
# For the 'duplicates', reading in the analyte info to get the SeqIDs

annot = pd.read_csv('SomalogicAnalyteInfoV1_anonymized.csv')

In [ ]:
# Creating a df with only the cases of interest

proteins_of_interest = annot[annot['entrez_gene_symbol'].isin(['ACHE', 'CTF1', 'LILRB2','PRSS8'])]

In [ ]:
proteins_of_interest

In [ ]:
# Creating a list of the seqIDs with established Olink correspondence- these will be more true to the sun et al study
# Only one of each protein duplicate has a seq id that directly corresponds to a protein panel- and they all have significant correlations
# The paper with the information is here: https://www.nature.com/articles/s41586-023-06563-x

seq_ids = ['seq_15513_108', 'seq_15553_22', 'seq_2889_37', 'seq_5091_28' , 'sample_id'] # sample id added for retrieval later

In [ ]:
df_no_isoforms.columns

In [ ]:
# Dropping all the proteins and then rerunning to extract just the proteins with the seqids I want

df_no_dups = df_no_isoforms.drop(columns= ['Acetylcholinesterase (ACHE)', 'Acetylcholinesterase (ACHE).1',  
                                           'Cardiotrophin-1 (CTF1)', 'Cardiotrophin-1 (CTF1).1' ,
                                           'Leukocyte immunoglobulin-like receptor subfamily B member 2 (LILRB2)',
                                           'Leukocyte immunoglobulin-like receptor subfamily B member 2 (LILRB2).1',
                                           'Prostasin (PRSS8)', 'Prostasin (PRSS8).1'])

In [ ]:
df_no_dups.columns

In [ ]:
# creating a list of file names to use for dataset access

files = [
    '~Somalogic01V1_anonymized.csv',
    'Somalogic02V1_anonymized.csv',
    'Somalogic03V1_anonymized.csv',
    'Somalogic04V1_anonymized.csv',
    'Somalogic05V1_anonymized.csv',
    'Somalogic06V1_anonymized.csv',
    'Somalogic07V1_anonymized.csv',
    'Somalogic08V1_anonymized.csv',
    'Somalogic09V1_anonymized.csv',
    'Somalogic10V1_anonymized.csv',
    'Somalogic11V1_anonymized.csv',
    'Somalogic12V1_anonymized.csv'
]

In [ ]:
# Iterating through files

wanted = seq_ids
combined = None
for file in files:
    df = pd.read_csv(file, usecols=lambda c: c in wanted)
    if len(df.columns) <= 1: # if no proteins found skip (aka for first few iterations before they find my proteins)
        continue
    combined = df if combined is None else combined.merge(df, on='sample_id', how='outer')

In [ ]:
combined

In [ ]:
labels = {
    row['column_name']: f"{row['target_full_name']} ({row['entrez_gene_symbol']})"
    for column, row in proteins_of_interest.iterrows()
}

combined = combined.rename(columns=labels)

In [ ]:
# Adding these back into the dataframe

df_final = df_no_dups = pd.merge(df_no_dups, combined, on= 'sample_id', how= 'left')

In [ ]:
df_complete = df_final.drop(columns = ['sample_id'])

In [ ]:
## Adding age squared term and age_x_sex interaction term to dataset

df_complete['age_squared'] = df_complete['age_at_visit'] ** 2
df_complete['age_x_sex'] = df_complete['age_at_visit'] * df_complete['sex']



In [ ]:
df_complete.to_csv('complete_df.csv', index= False)

In [ ]:
def short_name(label):
    if label.endswith(')') and '(' in label:
        return label[label.rfind('(') + 1 : -1]
    return label

In [ ]:
df_complete = df_complete.rename(columns=short_name)

In [ ]:
# Heatmap of feature correlations

# Select numeric columns only
numeric_df = df_complete.select_dtypes(include=['number'])

# Correlation matrix
corr_matrix = numeric_df.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)

plt.title('Feature Correlations')
plt.tight_layout()
plt.show()
plt.savefig('feature_correlations.svg', bbox_inches='tight')